# Feature hashing
Hashing maps arbitrary tokens into a fixed-dimensional sparse vector without learning a vocabulary. This is attractive when cardinality grows continuously and memory must remain bounded.

In [ ]:
import numpy as np
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng=np.random.default_rng(42); n=5000
records=[]; y=[]
for _ in range(n):
    city=f'city={rng.integers(0,500)}'; device=f'device={rng.choice(["ios","android","web"])}'; campaign=f'campaign={rng.integers(0,3000)}'
    records.append({city:1,device:1,campaign:1})
    y.append(int(device=='device=ios') ^ int(city in {'city=7','city=19','city=42'}))
hasher=FeatureHasher(n_features=2**12,input_type='dict',alternate_sign=False)
X=hasher.transform(records); y=np.array(y)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
m=SGDClassifier(loss='log_loss',max_iter=1500,random_state=42).fit(Xtr,ytr)
print('shape:',X.shape,'density:',round(X.nnz/(X.shape[0]*X.shape[1]),6),'AUC:',round(roc_auc_score(yte,m.predict_proba(Xte)[:,1]),4))


## Trade-off
Hash collisions are deliberate. More hash dimensions reduce collisions but cost memory. Unlike one-hot vocabularies, hashing handles unseen tokens naturally and avoids storing a dictionary — a core idea behind scalable systems such as Vowpal Wabbit.